# 05: Hybrid Model Training (Gradient Boosting)

**Objective:** Supervised training of XGBoost utilizing both tabular heuristics and deep sequence embeddings.

**Key Actions:**
- Merge (Join) Gold Layer heuristics with sequence embeddings on `NUEVO_ID`.
- Strict subsetting: Train ONLY on the ~1,023 labeled entities.
- Evaluate using predefined GroupKFold indexes (AUROC, F1-Score).

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# embeddings_df = pd.read_parquet('data/features/sequence_embeddings.parquet')
# gold_features = pd.read_parquet('data/gold/gold_heuristic_features.parquet')
# lookup_df = pd.read_parquet('data/silver/silver_layer.parquet')[['NUEVO_ID', 'ATSEG', 'fold']].drop_duplicates()

## 1. Feature Merging (Hybridization)

In [ ]:
def build_hybrid_dataset(embeddings: pd.DataFrame, heuristics: pd.DataFrame, lookup: pd.DataFrame) -> pd.DataFrame:
    """
    Joins the DL abstract features with the engineered heuristic features.
    """
    logger.info("Building hybrid feature matrix...")
    if embeddings.empty or heuristics.empty: return pd.DataFrame()
    
    df_final = pd.merge(heuristics, embeddings, on='NUEVO_ID', how='inner')
    df_final = pd.merge(df_final, lookup, on='NUEVO_ID', how='inner')
    
    logger.info(f"Hybrid Matrix shape: {df_final.shape}")
    return df_final

# hybrid_df = build_hybrid_dataset(embeddings_df, gold_features, lookup_df)

## 2. Model Training loop (GroupKFold strict partitions)

In [ ]:
def train_hybrid_xgboost(df: pd.DataFrame, target_col: str = 'ATSEG'):
    """
    Trains an XGBoost model recursively across the pre-defined folds ensuring zero leakage.
    """
    if df.empty: return
    
    # Filter exactly to labeled physicians (fold != -1)
    labeled_mask = df['fold'] != -1
    df_train = df[labeled_mask].copy()
    
    features = [c for c in df_train.columns if c not in ['NUEVO_ID', target_col, 'fold']]
    
    oof_preds = np.zeros(len(df_train))
    oof_targets = df_train[target_col].values
    folds = df_train['fold'].unique()
    
    models = []
    
    for f in folds:
        logger.info(f"--- Training Fold {f} ---")
        val_idx = df_train['fold'] == f
        trn_idx = df_train['fold'] != f
        
        X_trn, y_trn = df_train.loc[trn_idx, features], df_train.loc[trn_idx, target_col]
        X_val, y_val = df_train.loc[val_idx, features], df_train.loc[val_idx, target_col]
        
        clf = xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="auc",
            early_stopping_rounds=30
        )
        
        clf.fit(
            X_trn, y_trn,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        preds = clf.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = preds
        
        auc = roc_auc_score(y_val, preds)
        logger.info(f"Fold {f} AUROC: {auc:.4f}")
        models.append(clf)
        
    # Global Evaluation
    global_auc = roc_auc_score(oof_targets, oof_preds)
    hard_preds = (oof_preds > 0.5).astype(int)
    global_f1 = f1_score(oof_targets, hard_preds)
    
    logger.info(f"Overall Out-Of-Fold AUROC: {global_auc:.4f}")
    logger.info(f"Overall Out-Of-Fold F1-Score: {global_f1:.4f}")
    
    return models

# trained_models = train_hybrid_xgboost(hybrid_df)